# Learned dynamics and symmetry on the IDSIA nano-drone benchmark

This notebook uses the reproducible split from **Nonlinear System Identification for a Nano-drone Benchmark**: Square, Random, and Chirp runs 1–3 train the models; run 4 validates them; all Melon flights are held out for testing. Inputs are measured motor angular velocities and full motion-capture state at 100 Hz.

Every 50-step window is seeded once from the measured state. A GRU then predicts body-frame linear and angular acceleration using only its rolled-forward state and the motor sequence. Velocity and angular velocity are integrated from acceleration; position and SO(3) attitude are integrated from those velocities.

Three networks share the same recurrent acceleration architecture:

- **absolute:** rolled world position, world velocity, SO(3) rotation vector, body angular velocity, and motors;
- **canonical:** body gravity direction, body velocity, body angular velocity, and motors, removing absolute position and yaw;
- **symmetric:** exact recurrent equivariance under the left-right and front-back Crazyflie mirrors, including motor permutations and polar/axial output transformations.

Training runs for 20 epochs with a uniform 50-step full-state loss. Validation selects the checkpoint; the Melon test flights are untouched until final evaluation.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "src" / "learned_dynamics").exists() else HERE.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

from quadrotor.canonical import (
    ABSOLUTE_INPUTS,
    CANONICAL_INPUTS,
    absolute_dim,
    canonical_dim,
    output_dim,
)
from quadrotor.evaluate import benchmark_mae
from quadrotor.fetch_idsia import fetch
from quadrotor.idsia_data import load_idsia_prepared
from quadrotor.rollout import open_loop
from quadrotor.sequence import MotorNet
from quadrotor.train import train_open_loop

SEED = 0
EPOCHS = 20
LR = 1e-3
ROLLOUT_EVERY = 5
WINDOW_STRIDE = 1
BATCH_SIZE = 256

fetch()


def plot_rollouts(flights):
    print(f"{'flight':32s} {'absolute':>10s} {'canonical':>10s} {'symmetric':>10s}")
    cols = 3
    rows = int(np.ceil(len(flights) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 4.0 * rows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, flight in zip(axes, flights):
        absolute_path, absolute_rmse = open_loop(absolute_model, flight)
        canonical_path, canonical_rmse = open_loop(canonical_model, flight)
        symmetric_path, symmetric_rmse = open_loop(symmetric_model, flight)
        print(
            f"{flight.name:32s} {absolute_rmse:9.3f} m {canonical_rmse:9.3f} m {symmetric_rmse:9.3f} m"
        )
        ax.plot(flight.position[:, 0], flight.position[:, 1], color="black", lw=1.8, label="Vicon")
        ax.plot(absolute_path[:, 0], absolute_path[:, 1], color="C3", lw=1.3, label="absolute")
        ax.plot(canonical_path[:, 0], canonical_path[:, 1], color="C0", lw=1.3, label="canonical")
        ax.plot(symmetric_path[:, 0], symmetric_path[:, 1], color="C4", lw=1.3, label="symmetric")
        ax.set_title(flight.name.replace("_", " "))
        ax.set_aspect("equal", adjustable="datalim")
        ax.set_xlabel("x (m)")
        ax.set_ylabel("y (m)")
        ax.grid(alpha=0.25)
        ax.legend(loc="best", fontsize=8)
    for ax in axes[len(flights) :]:
        ax.axis("off")
    plt.show()


NanoBench already present: 107 csv files in /Users/antony/Desktop/GTech_FA26/capstone/learned-dynamics-and-symmetry/data/nanobench/datasets/dataset


In [2]:
class AbsoluteModel(MotorNet):
    """GRU acceleration model with absolute coordinates.

    Its recurrent input contains its own rolled-forward position, world
    velocity, SO(3) rotation vector, body angular velocity, and motor commands.
    It outputs body linear and angular acceleration only.
    """

    reads_pose = True

    def __init__(self):
        super().__init__(absolute_dim(), output_dim(), mirror=False)


class CanonicalModel(MotorNet):
    """GRU acceleration model with position and yaw canonicalized away.

    The body gravity direction represents tilt without Euler singularities.
    Body velocity, body angular velocity, and motor commands complete the input.
    """

    def __init__(self):
        super().__init__(canonical_dim(), output_dim(), mirror=False)


class SymmetricModel(MotorNet):
    """Canonical recurrent model with exact Crazyflie mirror equivariance.

    Four shared-weight GRU branches process the identity, left-right,
    front-back, and combined reflections. Their body acceleration predictions
    are transformed back and averaged at every rollout step.
    """

    def __init__(self):
        super().__init__(canonical_dim(), output_dim(), mirror=True)


In [3]:
train_flights, val_flights, test_flights = load_idsia_prepared()
print("train, Square/Random/Chirp runs 1-3:")
for flight in train_flights:
    print(f"  {flight.name:42s}  samples {flight.n_poses:5d}")
print("validation, Square/Random/Chirp run 4:")
for flight in val_flights:
    print(f"  {flight.name:42s}  samples {flight.n_poses:5d}")
print("test, held-out Melon flights:")
for flight in test_flights:
    print(f"  {flight.name:42s}  samples {flight.n_poses:5d}")
print("absolute   " + ", ".join(ABSOLUTE_INPUTS))
print("canonical  " + ", ".join(CANONICAL_INPUTS))


train, first 80% of each multisine:
  A1b_multisine_sysid_rep1 train              samples  5779  attitude vicon
  A1b_multisine_sysid_rep2 train              samples  5765  attitude vicon
validation, last 20%:
  A1b_multisine_sysid_rep1 val                samples  1445  attitude vicon
  A1b_multisine_sysid_rep2 val                samples  1442  attitude vicon
test, Category B: 103 flights
absolute   x, y, z, vx_world, vy_world, vz_world, rotation_x, rotation_y, rotation_z, omega_x_body, omega_y_body, omega_z_body, m1, m2, m3, m4
canonical  roll, pitch, vx_body, vy_body, vz_body, omega_x_body, omega_y_body, omega_z_body, m1, m2, m3, m4


In [ ]:
torch.manual_seed(SEED)
absolute_model = AbsoluteModel()
absolute_result = train_open_loop(
    absolute_model,
    train_flights,
    val_flights,
    test_flights,
    epochs=EPOCHS,
    lr=LR,
    rollout_every=ROLLOUT_EVERY,
    batch_size=BATCH_SIZE,
    stride=WINDOW_STRIDE,
    name="absolute",
)
absolute_history = absolute_result.history

torch.manual_seed(SEED)
canonical_model = CanonicalModel()
canonical_result = train_open_loop(
    canonical_model,
    train_flights,
    val_flights,
    test_flights,
    epochs=EPOCHS,
    lr=LR,
    rollout_every=ROLLOUT_EVERY,
    batch_size=BATCH_SIZE,
    stride=WINDOW_STRIDE,
    name="canonical",
)
canonical_history = canonical_result.history

torch.manual_seed(SEED)
symmetric_model = SymmetricModel()
symmetric_result = train_open_loop(
    symmetric_model,
    train_flights,
    val_flights,
    test_flights,
    epochs=EPOCHS,
    lr=LR,
    rollout_every=ROLLOUT_EVERY,
    batch_size=BATCH_SIZE,
    stride=WINDOW_STRIDE,
    name="symmetric",
)
symmetric_history = symmetric_result.history


absolute    50 steps  stride 1  train 11444  val 2787  test flights 103  batch 256
absolute    epoch 000  weighted mse 5.524352e-01  val 4.474123e-01  h10 MAE p 0.0403 m v 0.1442 m/s R 0.1068 rad w 1.8364 rad/s  h50 MAE p 0.1787 m v 0.4727 m/s R 0.2135 rad w 1.8066 rad/s  saved
absolute    epoch 001  weighted mse 4.216861e-01  val 2.786846e-01  saved
absolute    epoch 002  weighted mse 2.111185e-01  val 2.224277e-01  saved
absolute    epoch 003  weighted mse 1.602789e-01  val 2.229286e-01
absolute    epoch 004  weighted mse 1.280512e-01  val 2.234169e-01
absolute    epoch 005  weighted mse 1.089557e-01  val 2.357835e-01
absolute    epoch 006  weighted mse 9.641639e-02  val 2.329494e-01
absolute    epoch 007  weighted mse 8.913299e-02  val 2.364066e-01
absolute    epoch 008  weighted mse 8.238900e-02  val 2.318712e-01
absolute    epoch 009  weighted mse 7.615488e-02  val 2.405142e-01
absolute    epoch 010  weighted mse 7.224895e-02  val 2.459930e-01  h10 MAE p 0.0413 m v 0.1109 m/s R 0.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.4), constrained_layout=True)
for history, color, label in (
    (absolute_history, "C3", "absolute"),
    (canonical_history, "C0", "canonical"),
    (symmetric_history, "C4", "symmetric"),
):
    epochs = [row["epoch"] for row in history]
    ax.plot(epochs, [row["train_loss"] for row in history], color=color, label=f"{label} train")
    ax.plot(
        epochs,
        [row["val_loss"] for row in history],
        color=color,
        ls="--",
        label=f"{label} val",
    )
ax.set_xlabel("epoch")
ax.set_ylabel("weighted standardized state MSE")
ax.set_yscale("log")
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
plt.show()


Full-flight stress tests on the three held-out Melon recordings. Each model receives one initial motion-capture state and then only the measured motor-speed sequence. This is stricter and much longer than the official 0.5-second benchmark.


In [ ]:
print(f"{'flight':32s} {'absolute':>10s} {'canonical':>10s} {'symmetric':>10s}")
for flight in test_flights:
    absolute_rmse = open_loop(absolute_model, flight)[1]
    canonical_rmse = open_loop(canonical_model, flight)[1]
    symmetric_rmse = open_loop(symmetric_model, flight)[1]
    print(
        f"{flight.name:32s} {absolute_rmse:9.3f} m "
        f"{canonical_rmse:9.3f} m {symmetric_rmse:9.3f} m"
    )
plot_rollouts(test_flights)


Full-flight rollouts on the nine Square, Random, and Chirp training recordings. Validation run 4 is not drawn here.


In [ ]:
plot_rollouts(train_flights)


Official IDSIA-style multi-step metrics on every valid window from the three held-out Melon flights: vector-norm MAE at 1, 10, and 50 steps, plus cumulative error over h=1…50.


In [ ]:
benchmark_scores = {}
for name, network in (
    ("absolute", absolute_model),
    ("canonical", canonical_model),
    ("symmetric", symmetric_model),
):
    print(f"evaluating {name} on all held-out Melon windows", flush=True)
    benchmark_scores[name] = benchmark_mae(network, test_flights, batch_size=BATCH_SIZE)

print(f"{'model':12s} {'state':10s} {'h=1':>10s} {'h=10':>10s} {'h=50':>10s} {'sum 1:50':>12s}")
for name, score in benchmark_scores.items():
    for state in ("position", "velocity", "rotation", "omega"):
        value = score[state]
        print(
            f"{name:12s} {state:10s} {value[0]:10.4f} {value[9]:10.4f} "
            f"{value[49]:10.4f} {value.sum():12.4f}"
        )
